# 07. (선택 과제) 실시간 추론 웹 애플리케이션

> **선택/심화 과제입니다.** 배포한 SageMaker 엔드포인트를 호출하는 **웹앱**을 만들어 봅니다.
> **Gradio**를 사용하며 `share=True` 옵션으로 공개 URL이 자동 생성되어 별도 포트/프록시 설정 없이 바로 접속할 수 있습니다.

## 아키텍처
```
브라우저 ──HTTPS──> Gradio 공개 URL ──> 노트북 내 Gradio 서버 ──boto3/SigV4──> SageMaker 엔드포인트
```
- 웹앱은 폼 입력을 36개 피처 벡터로 조립해 엔드포인트에 보내고, 3개 클래스 확률을 받아 표시합니다.
- 엔드포인트는 공개되지 않으며 **AWS 자격증명(IAM)** 으로만 호출됩니다.

## 전제
`05_deployment.ipynb` 로 엔드포인트가 떠 있어야 합니다.

## 0. Gradio 설치 & 엔드포인트 상태 확인

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "-qU"],
                      stdout=subprocess.DEVNULL)
print("gradio installed")

import boto3, sagemaker
sess = sagemaker.Session()
%store -r endpoint_name region
sm = boto3.client("sagemaker", region_name=region)
status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
print(f"{endpoint_name} -> {status}")
assert status == "InService", "엔드포인트가 InService 상태가 아닙니다."

## 1. 웹앱 실행

아래 셀에서 `call_endpoint` 함수를 완성하세요. 나머지 UI 코드는 제공됩니다.
셀을 실행하면 Gradio가 공개 URL을 출력합니다. 해당 URL을 브라우저에서 열면 됩니다.

> `share=True`로 생성된 URL은 **72시간** 동안 유효하며 노트북 커널이 살아 있는 동안만 작동합니다.

In [ ]:
import json
import boto3
import numpy as np
import gradio as gr

%store -r endpoint_name region

CLASSES = ["Dropout", "Enrolled", "Graduate"]
runtime = boto3.client("sagemaker-runtime", region_name=region)

FEATURES = [
    "Marital status", "Application mode", "Application order", "Course",
    "Daytime/evening attendance", "Previous qualification", "Previous qualification (grade)",
    "Nacionality", "Mother\'s qualification", "Father\'s qualification", "Mother\'s occupation",
    "Father\'s occupation", "Admission grade", "Displaced", "Educational special needs",
    "Debtor", "Tuition fees up to date", "Gender", "Scholarship holder", "Age at enrollment",
    "International", "Curricular units 1st sem (credited)", "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)", "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)", "Curricular units 1st sem (without evaluations)",
    "Curricular units 2nd sem (credited)", "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)", "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)", "Curricular units 2nd sem (without evaluations)",
    "Unemployment rate", "Inflation rate", "GDP",
]

DEFAULTS = {
    "Marital status": 1, "Application mode": 17, "Application order": 1, "Course": 9238,
    "Daytime/evening attendance": 1, "Previous qualification": 1, "Previous qualification (grade)": 133.1,
    "Nacionality": 1, "Mother\'s qualification": 19, "Father\'s qualification": 19,
    "Mother\'s occupation": 5, "Father\'s occupation": 7, "Admission grade": 126.1,
    "Displaced": 1, "Educational special needs": 0, "Debtor": 0, "Tuition fees up to date": 1,
    "Gender": 0, "Scholarship holder": 0, "Age at enrollment": 20, "International": 0,
    "Curricular units 1st sem (credited)": 0, "Curricular units 1st sem (enrolled)": 6,
    "Curricular units 1st sem (evaluations)": 8, "Curricular units 1st sem (approved)": 5,
    "Curricular units 1st sem (grade)": 12.286, "Curricular units 1st sem (without evaluations)": 0,
    "Curricular units 2nd sem (credited)": 0, "Curricular units 2nd sem (enrolled)": 6,
    "Curricular units 2nd sem (evaluations)": 8, "Curricular units 2nd sem (approved)": 5,
    "Curricular units 2nd sem (grade)": 12.2, "Curricular units 2nd sem (without evaluations)": 0,
    "Unemployment rate": 11.1, "Inflation rate": 1.4, "GDP": 0.32,
}


def call_endpoint(feature_row):
    # TODO: 06 노트북의 boto3 호출 방식을 응용해 엔드포인트를 호출하세요.
    # feature_row(리스트)를 CSV 문자열로 변환 -> invoke_endpoint -> 응답 파싱 -> 확률 리스트 반환
    raise NotImplementedError("call_endpoint 를 구현하세요")


def predict(age, admission, prev_grade, gender, s1_approved, s1_grade, s2_approved, s2_grade,
            tuition, scholarship, debtor):
    overrides = {
        "Age at enrollment": age, "Admission grade": admission,
        "Previous qualification (grade)": prev_grade, "Gender": gender,
        "Curricular units 1st sem (approved)": s1_approved,
        "Curricular units 1st sem (grade)": s1_grade,
        "Curricular units 2nd sem (approved)": s2_approved,
        "Curricular units 2nd sem (grade)": s2_grade,
        "Tuition fees up to date": int(tuition), "Scholarship holder": int(scholarship),
        "Debtor": int(debtor),
    }
    values = dict(DEFAULTS)
    values.update(overrides)
    feature_row = [values[f] for f in FEATURES]
    proba = call_endpoint(feature_row)
    pred = CLASSES[int(np.argmax(proba))]
    result = {CLASSES[i]: round(proba[i], 4) for i in range(3)}
    return pred, result


demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Slider(17, 60, value=20, step=1, label="Age at enrollment"),
        gr.Slider(0, 200, value=126.1, step=0.1, label="Admission grade"),
        gr.Slider(0, 200, value=133.1, step=0.1, label="Previous qualification grade"),
        gr.Radio(["Female (0)", "Male (1)"], value="Female (0)", label="Gender", type="index"),
        gr.Slider(0, 26, value=5, step=1, label="1st sem approved units"),
        gr.Slider(0, 20, value=12.3, step=0.1, label="1st sem grade"),
        gr.Slider(0, 26, value=5, step=1, label="2nd sem approved units"),
        gr.Slider(0, 20, value=12.2, step=0.1, label="2nd sem grade"),
        gr.Checkbox(value=True, label="Tuition fees up to date"),
        gr.Checkbox(value=False, label="Scholarship holder"),
        gr.Checkbox(value=False, label="Debtor"),
    ],
    outputs=[
        gr.Textbox(label="Prediction"),
        gr.Label(label="Class probabilities"),
    ],
    title="Student Success Prediction",
    description=f"SageMaker Endpoint: {endpoint_name} | Region: {region}",
)
demo.launch(share=True)

## 2. 종료

테스트가 끝나면 아래 셀을 실행해 Gradio 서버를 종료합니다.

In [ ]:
demo.close()
print("Gradio server stopped")

🎉 **수고하셨습니다!** 데이터 전처리부터 학습·평가·튜닝·배포·추론, 그리고 웹앱까지 완성했습니다.

> 워크샵이 끝나면 `08_cleanup.ipynb` 를 실행해 모든 리소스를 정리하세요.